In [33]:
import uuid
import pandas as pd
import psycopg2
import sys
import os
from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.models import Distance, VectorParams, SparseVectorParams
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
sys.path.append(os.path.abspath(os.path.join('..')))
from final_project.connect_to_google_drive import get_sheets_client, SHEET_ID
from fastembed import SparseTextEmbedding

In [16]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

In [17]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

In [19]:
sheets_client = get_sheets_client()
sheet = sheets_client.open_by_key(SHEET_ID).sheet1
df = pd.DataFrame(sheet.get_all_records())
to_sync = df[(df['status'] == 'Success') & (df['vector_db_sync'] == 'No')]

In [20]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [21]:
def get_text_from_postgres(file_id):
    try:
        conn = psycopg2.connect(
            dbname="ucu_rag_db", user="user", password="password", host="127.0.0.1", port=5432
        )
        cur = conn.cursor()
        cur.execute(
            "SELECT markdown_content FROM processed_documents WHERE google_drive_id = %s", 
            (file_id,)
        )
        result = cur.fetchone()
        cur.close()
        conn.close()
        return result[0] if result else None
    except Exception as e:
        print(f"ERROR: {e}")
        return None

In [40]:
def upload_to_qdrant(to_sync, model, collection, chunking_method, e5=False, sparse_model=None):
    for _, row in to_sync.iterrows():
            file_id = row['google_drive_id']
            file_name = row['file_name']
            
            text = get_text_from_postgres(file_id)

            if text:
                if chunking_method == "fixed_size":
                    chunks = [text[i:i+1000] for i in range(0, len(text), 800)]
                elif chunking_method == "recursive_character":
                    chunks = text_splitter.split_text(text)
                for i, chunk in enumerate(chunks):
                    if e5:
                        dense_vector = model.encode("passage: " + chunk).tolist()
                    else:
                        dense_vector = model.encode(chunk).tolist()
                    vectors_to_upload = {"default": dense_vector}
                    if sparse_model:
                        sparse_emb = list(sparse_model.embed([chunk]))[0]
                        vectors_to_upload["text_sparse"] = models.SparseVector(
                            indices=sparse_emb.indices.tolist(),
                            values=sparse_emb.values.tolist()
                        )
                    point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{file_id}_{i}"))
                    client.upsert(
                        collection_name=collection,
                        points=[
                            models.PointStruct(
                                id=point_id,
                                vector=vectors_to_upload,
                                payload={
                                    "file_id": file_id,
                                    "file_name": file_name,
                                    "text": chunk,
                                    "chunk_index": i
                                }
                            )
                        ]
                    )
                print(f"File {file_name} uploaded into Qdrant")

<hr>

# paraphrase-multilingual-MiniLM-L12-v2

In [23]:
MODEL_BASELINE = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4360.28it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
COLLECTION_BASELINE = "ucu_documents_baseline"

In [25]:
if not client.collection_exists(COLLECTION_BASELINE):
    client.create_collection(
        collection_name=COLLECTION_BASELINE,
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
    )

In [ ]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_BASELINE, collection=COLLECTION_BASELINE, chunking_method="fixed_size")

File plagiarism_check_policy.pdf uploaded into Qdrant
File academic_integrity_policy.pdf uploaded into Qdrant
File Plan-rozvytku-polityky-rivnosti-ta-inklyuzyvnogo-robochogo-seredovyshhe-v-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-Otsinku-efektyvnosti-pratsivnykiv-Ukrayinskogo-katolytskogo-Universytetu-1.pdf uploaded into Qdrant
File Polozhennya-pro-pidbir-pratsivnykiv-Ukrayinskogo-katolytskogo-universytetu-1.pdf uploaded into Qdrant
File Pravyla-vnutrishnogo-trudovogo-rozporyadku-1.pdf uploaded into Qdrant
File Polozhennya-pro-sluzhbovi-vidryadzhennya-personalu-ZVO-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-dystantsijnu-robotu-ta-gnuchkyj-rezhym-robochogo-chasu-Ukrayinskogo-katolytskogo-universytetu-1.pdf uploaded into Qdrant
File Polozhennya-pro-shhorichnu-vidznaku-Vykladach-roku.pdf uploaded into Qdrant
File Poryadok-realizatsiyi-programy-rozvytku-molodyh-naukovo-pedagogichnyh-ta-pedagogichnyh-pratsivnykiv-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-poryadok-vyz

In [30]:
COLLECTION_BASELINE_RECURSIVE_CHAR = "ucu_documents_baseline_recursive_char"

In [31]:
if not client.collection_exists(COLLECTION_BASELINE_RECURSIVE_CHAR):
    client.create_collection(
        collection_name=COLLECTION_BASELINE_RECURSIVE_CHAR,
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
    )

In [ ]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_BASELINE, collection=COLLECTION_BASELINE_RECURSIVE_CHAR, chunking_method="recursive_character")

File plagiarism_check_policy.pdf uploaded into Qdrant
File academic_integrity_policy.pdf uploaded into Qdrant
File Plan-rozvytku-polityky-rivnosti-ta-inklyuzyvnogo-robochogo-seredovyshhe-v-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-Otsinku-efektyvnosti-pratsivnykiv-Ukrayinskogo-katolytskogo-Universytetu-1.pdf uploaded into Qdrant
File Polozhennya-pro-pidbir-pratsivnykiv-Ukrayinskogo-katolytskogo-universytetu-1.pdf uploaded into Qdrant
File Pravyla-vnutrishnogo-trudovogo-rozporyadku-1.pdf uploaded into Qdrant
File Polozhennya-pro-sluzhbovi-vidryadzhennya-personalu-ZVO-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-dystantsijnu-robotu-ta-gnuchkyj-rezhym-robochogo-chasu-Ukrayinskogo-katolytskogo-universytetu-1.pdf uploaded into Qdrant
File Polozhennya-pro-shhorichnu-vidznaku-Vykladach-roku.pdf uploaded into Qdrant
File Poryadok-realizatsiyi-programy-rozvytku-molodyh-naukovo-pedagogichnyh-ta-pedagogichnyh-pratsivnykiv-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-poryadok-vyz

<hr>

## Add sparse vectors for hybrid search

In [37]:
MODEL_SPARSE = SparseTextEmbedding(model_name="Qdrant/bm25")

In [42]:
COLLECTION_BASELINE_HYBRID = "ucu_documents_baseline_hybrid"

In [43]:
if not client.collection_exists(COLLECTION_BASELINE_HYBRID):
    client.create_collection(
        collection_name=COLLECTION_BASELINE_HYBRID,
        vectors_config={
            "default": VectorParams(
                size=384,
                distance=Distance.COSINE
            )
        },
        sparse_vectors_config={
            "text_sparse": SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=False,
                )
            )
        }
    )

In [44]:
upload_to_qdrant(to_sync=to_sync, model=MODEL_BASELINE, collection=COLLECTION_BASELINE_HYBRID, chunking_method="recursive_character", sparse_model=MODEL_SPARSE)

File plagiarism_check_policy.pdf uploaded into Qdrant
File academic_integrity_policy.pdf uploaded into Qdrant
File Plan-rozvytku-polityky-rivnosti-ta-inklyuzyvnogo-robochogo-seredovyshhe-v-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-Otsinku-efektyvnosti-pratsivnykiv-Ukrayinskogo-katolytskogo-Universytetu-1.pdf uploaded into Qdrant
File Polozhennya-pro-pidbir-pratsivnykiv-Ukrayinskogo-katolytskogo-universytetu-1.pdf uploaded into Qdrant
File Pravyla-vnutrishnogo-trudovogo-rozporyadku-1.pdf uploaded into Qdrant
File Polozhennya-pro-sluzhbovi-vidryadzhennya-personalu-ZVO-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-dystantsijnu-robotu-ta-gnuchkyj-rezhym-robochogo-chasu-Ukrayinskogo-katolytskogo-universytetu-1.pdf uploaded into Qdrant
File Polozhennya-pro-shhorichnu-vidznaku-Vykladach-roku.pdf uploaded into Qdrant
File Poryadok-realizatsiyi-programy-rozvytku-molodyh-naukovo-pedagogichnyh-ta-pedagogichnyh-pratsivnykiv-UKU.pdf uploaded into Qdrant
File Polozhennya-pro-poryadok-vyz